# Can we predict late deliveries?
Target: `late_delivery` (1 = delivered after the promised date). Goal: judge whether features known **at dispatch** predict lateness, versus a naive baseline.

In [ ]:
import os
os.environ.setdefault('MPLCONFIGDIR', os.environ.get('TMPDIR', '/tmp'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, RocCurveDisplay, PrecisionRecallDisplay
from sklearn.inspection import permutation_importance

df = pd.read_csv('data.csv', parse_dates=['dispatched_on'])
print(df.shape, '|', df.isna().sum().sum(), 'missing |', df.consignment.duplicated().sum(), 'duplicate ids')
df.describe(include='all').T

## 1. Target and a leakage check

In [ ]:
print(df.late_delivery.value_counts(normalize=True).round(3))
df.groupby(df.claim_paid_eur > 0).late_delivery.agg(['mean', 'size'])

**`claim_paid_eur` is excluded as a feature.** It is compensation paid on a claim *settled for the consignment*, so it only exists after delivery. The table shows it nearly encodes the target: no consignment without a claim is late, and ~90% of those with a claim are. A model using it would look excellent but be useless at dispatch time.

Features used (all known at dispatch): `carrier`, `distance_km`, `weight_kg`, `parcels`, and hour / day-of-week from `dispatched_on`. `consignment` is only an ID.

## 2. Exploration

In [ ]:
d = df.copy()
d['hour'] = d.dispatched_on.dt.hour; d['dow'] = d.dispatched_on.dt.dayofweek; d['month'] = d.dispatched_on.dt.to_period('M').astype(str)
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
d.groupby('carrier').late_delivery.mean().plot.bar(ax=ax[0, 0], title='By carrier')
d.groupby(pd.qcut(d.distance_km, 10, duplicates='drop'), observed=True).late_delivery.mean().plot(ax=ax[0, 1], marker='o', title='By distance decile'); ax[0, 1].tick_params(axis='x', rotation=60)
d.groupby(pd.qcut(d.weight_kg, 10, duplicates='drop'), observed=True).late_delivery.mean().plot(ax=ax[0, 2], marker='o', title='By weight decile'); ax[0, 2].tick_params(axis='x', rotation=60)
d.groupby('parcels').late_delivery.mean().plot.bar(ax=ax[1, 0], title='By parcels')
d.groupby('hour').late_delivery.mean().plot(ax=ax[1, 1], marker='o', title='By dispatch hour')
d.groupby('month').late_delivery.mean().plot(ax=ax[1, 2], marker='o', title='By dispatch month'); ax[1, 2].tick_params(axis='x', rotation=60)
for a in ax.ravel(): a.set_ylabel('late rate'); a.set_xlabel('')
plt.tight_layout(); plt.show()

## 3. Late rate over time: a data-quality warning

In [ ]:
m = d.groupby('month').agg(n=('late_delivery', 'size'), late_rate=('late_delivery', 'mean'), claim_rate=('claim_paid_eur', lambda x: (x > 0).mean()))
m.plot(y=['late_rate', 'claim_rate'], marker='o', figsize=(10, 3.5), title='Monthly late rate and claim rate'); plt.xticks(rotation=60); plt.show()
d.groupby(['month', 'carrier']).late_delivery.mean().unstack().round(3).tail(8)

The late rate swings seasonally (about 10% to 49% month to month) but from **October 2025 it collapses to ~1-5% in every carrier at once**, with the claim rate falling alongside. A simultaneous drop across all carriers is more consistent with **labels that are not yet complete for recent dispatches** (late deliveries or claims still to be recorded) than with a real operational improvement, though the data alone cannot prove which. Either way, those months are a poor test bed: a model scored on them is judged against labels that behave differently from the training period.

**Assumption:** treat consignments dispatched from `2025-10-01` as immature and exclude them from the main evaluation (`CUTOFF` below). Section 6 repeats the evaluation on all data as a sensitivity check. Please confirm with the data owner how `late_delivery` is populated for recent shipments.

## 4. Modelling
We will predict *future* consignments, so we split by time: train on the earliest 70%, validate on the next 15%, and keep the latest 15% (of the mature period) as an untouched test set. Metrics: ROC AUC and average precision, against a base-rate dummy.

In [ ]:
CUTOFF = pd.Timestamp('2025-10-01')
full = d.sort_values('dispatched_on').reset_index(drop=True)
d = full[full.dispatched_on < CUTOFF].reset_index(drop=True)
n = len(d); i1, i2 = int(n * .70), int(n * .85)
d['cal_month'] = d.dispatched_on.dt.month; full['cal_month'] = full.dispatched_on.dt.month
num = ['distance_km', 'weight_kg', 'parcels', 'hour', 'dow', 'cal_month']; cat = ['carrier']
X = d[num + cat]; y = d.late_delivery
Xtr, ytr, Xva, yva, Xte, yte = X[:i1], y[:i1], X[i1:i2], y[i1:i2], X[i2:], y[i2:]
print('train', d.dispatched_on[0].date(), '->', d.dispatched_on[i1 - 1].date(), '| test from', d.dispatched_on[i2].date())
print('late rate train/val/test:', ytr.mean().round(3), yva.mean().round(3), yte.mean().round(3))

pre_gb = ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cat)], remainder='passthrough')
pre_lr = ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cat), ('n', StandardScaler(), num)])
models = {
    'baseline (base rate)': DummyClassifier(strategy='prior'),
    'logistic regression': make_pipeline(pre_lr, LogisticRegression(max_iter=1000)),
    'gradient boosting': make_pipeline(pre_gb, HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)),
}
rows = []
for name, m in models.items():
    m.fit(Xtr, ytr)
    for split, (Xs, ys) in {'val': (Xva, yva), 'test': (Xte, yte)}.items():
        p = m.predict_proba(Xs)[:, 1]
        rows.append(dict(model=name, split=split, roc_auc=roc_auc_score(ys, p), avg_precision=average_precision_score(ys, p), brier=brier_score_loss(ys, p)))
pd.DataFrame(rows).round(3).pivot(index='model', columns='split')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, m in models.items():
    p = m.predict_proba(Xte)[:, 1]
    RocCurveDisplay.from_predictions(yte, p, name=name, ax=ax[0])
    PrecisionRecallDisplay.from_predictions(yte, p, name=name, ax=ax[1])
ax[0].set_title('ROC (test)'); ax[1].set_title('Precision-recall (test)'); plt.tight_layout(); plt.show()

## 5. What drives the prediction?

In [ ]:
best = models['gradient boosting']
pi = permutation_importance(best, Xte, yte, scoring='roc_auc', n_repeats=20, random_state=0)
s = pd.Series(pi.importances_mean, index=Xte.columns).sort_values()
s.plot.barh(xerr=pd.Series(pi.importances_std, index=Xte.columns)[s.index])
plt.xlabel('drop in test ROC AUC when shuffled'); plt.title('Permutation importance'); plt.show()

## 6. Does predicted risk rank real risk?

In [ ]:
t = pd.DataFrame({'p': best.predict_proba(Xte)[:, 1], 'y': yte.values})
t['risk_quintile'] = pd.qcut(t.p, 5, labels=False, duplicates='drop')
t.groupby('risk_quintile').agg(mean_pred=('p', 'mean'), observed_late=('y', 'mean'), n=('y', 'size')).round(3)

## 7. Sensitivity: all data, including the post-October-2025 months

In [ ]:
n2 = len(full); k = int(n2 * .70); j = int(n2 * .85)
Xf, yf = full[num + cat], full.late_delivery
rows = []
for name, m in models.items():
    m.fit(Xf[:k], yf[:k])
    p = m.predict_proba(Xf[j:])[:, 1]
    rows.append(dict(model=name, test_late_rate=yf[j:].mean(), roc_auc=roc_auc_score(yf[j:], p), avg_precision=average_precision_score(yf[j:], p)))
pd.DataFrame(rows).round(3)